** Question 1 :**
Use Github Copilot to create a table SalesData whose columns are CustomerID, Name, Age, City,
PurchaseAmount, PurchaseDate and then ask to insert 10,000 rows of random data.

In [ ]:
-- Check existence, create database if missing, with basic error handling
IF DB_ID(N'copilot') IS NULL
BEGIN
    BEGIN TRY
        CREATE DATABASE [copilot];
        PRINT 'Database [copilot] created.';
    END TRY
    BEGIN CATCH
        SELECT
            ERROR_NUMBER()   AS ErrorNumber,
            ERROR_SEVERITY() AS Severity,
            ERROR_STATE()    AS State,
            ERROR_LINE()     AS Line,
            ERROR_MESSAGE()  AS Message;
    END CATCH;
END
ELSE
    PRINT 'Database [copilot] already exists.';
GO

IF DB_ID(N'copilot') IS NOT NULL
BEGIN
    BEGIN TRY
        USE [copilot];
        PRINT 'Database context set to [copilot].';
    END TRY
    BEGIN CATCH
        SELECT ERROR_NUMBER() AS ErrorNumber, ERROR_MESSAGE() AS Message;
    END CATCH;
END
ELSE
    PRINT 'Database [copilot] does not exist.';
GO

-- 1) Create table if it doesn't exist
IF OBJECT_ID(N'dbo.SalesData', N'U') IS NULL
BEGIN
    BEGIN TRY
        CREATE TABLE dbo.SalesData
        (
            CustomerID INT IDENTITY(1,1) PRIMARY KEY,
            [Name] NVARCHAR(100) NOT NULL,
            Age TINYINT NULL,
            City NVARCHAR(100) NULL,
            PurchaseAmount DECIMAL(10,2) NOT NULL,
            PurchaseDate DATETIME2 NOT NULL
        );
        PRINT 'Table dbo.SalesData created.';
    END TRY
    BEGIN CATCH
        SELECT ERROR_NUMBER() AS ErrorNumber, ERROR_SEVERITY() AS Severity,
               ERROR_STATE() AS State, ERROR_LINE() AS Line, ERROR_MESSAGE() AS Message;
    END CATCH;
END
ELSE
    PRINT 'Table dbo.SalesData already exists.';
GO

-- 2) Insert 10,000 rows of randomized sample data (run only if you confirm)
BEGIN TRY
    BEGIN TRAN;

    ;WITH
    E(n) AS (SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 1
             UNION ALL SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 1 UNION ALL SELECT 1), -- 10
    Tally AS (
        SELECT TOP (10000) ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS rn
        FROM E a CROSS JOIN E b CROSS JOIN E c CROSS JOIN E d CROSS JOIN E e
    )
    INSERT INTO dbo.SalesData (Name, Age, City, PurchaseAmount, PurchaseDate)
    SELECT
        -- Simple pseudo-name to avoid large lookup lists
        CASE ABS(CHECKSUM(NEWID())) % 10
            WHEN 0 THEN N'John' WHEN 1 THEN N'Mary' WHEN 2 THEN N'Alice' WHEN 3 THEN N'Bob' WHEN 4 THEN N'Carol'
            WHEN 5 THEN N'David' WHEN 6 THEN N'Eve' WHEN 7 THEN N'Frank' WHEN 8 THEN N'Grace' ELSE N'Hank' END
        + N' ' + LEFT(CONVERT(NVARCHAR(36), NEWID()), 8) AS [Name],
        CAST(ABS(CHECKSUM(NEWID())) % 60 + 18 AS TINYINT) AS Age, -- ages 18-77
        CASE ABS(CHECKSUM(NEWID())) % 10
            WHEN 0 THEN N'New York' WHEN 1 THEN N'Los Angeles' WHEN 2 THEN N'Chicago' WHEN 3 THEN N'Houston' WHEN 4 THEN N'Phoenix'
            WHEN 5 THEN N'Philadelphia' WHEN 6 THEN N'San Antonio' WHEN 7 THEN N'San Diego' WHEN 8 THEN N'Dallas' ELSE N'San Jose' END AS City,
        CONVERT(DECIMAL(10,2), (ABS(CHECKSUM(NEWID())) % 100000) / 100.0) AS PurchaseAmount, -- 0.00 - 1000.00
        DATEADD(DAY, - (ABS(CHECKSUM(NEWID())) % 3650), SYSDATETIME()) AS PurchaseDate -- within ~10 years
    FROM Tally;

    COMMIT TRAN;
    PRINT 'Inserted 10,000 rows into dbo.SalesData.';
END TRY
BEGIN CATCH
    IF XACT_STATE() <> 0 ROLLBACK TRAN;
    SELECT ERROR_NUMBER() AS ErrorNumber, ERROR_MESSAGE() AS Message;
END CATCH;
GO

IF OBJECT_ID(N'dbo.SalesData', N'U') IS NOT NULL
    SELECT * FROM dbo.SalesData;
ELSE
    PRINT 'Table dbo.SalesData does not exist in the current database.';

USE [copilot];
SELECT * FROM dbo.SalesData;
SELECT count(*) FROM dbo.SalesData;

**Question 2 :**
Use Copilot to:

Find total sales per city

Top 5 cities by revenue

In [ ]:
# Find total sales per city
SELECT City, SUM(PurchaseAmount) AS TotalSales, COUNT_BIG(*) AS Orders
FROM copilot.dbo.SalesData
GROUP BY City
ORDER BY TotalSales DESC;

In [ ]:
# Top 5 cities by revenue
SELECT TOP (5) City, SUM(PurchaseAmount) AS TotalSales, COUNT_BIG(*) AS Orders
FROM copilot.dbo.SalesData
GROUP BY City
ORDER BY TotalSales DESC;

**Question 3**  : Ask Copilot:

Find customers with purchases above average

In [ ]:
# Find customers with purchases above average
WITH CustomerTotals AS (
    SELECT CustomerID, [Name], SUM(PurchaseAmount) AS TotalPurchases
    FROM copilot.dbo.SalesData
    GROUP BY CustomerID, [Name]
)
SELECT CustomerID, [Name], TotalPurchases
FROM CustomerTotals
WHERE TotalPurchases > (SELECT AVG(TotalPurchases) FROM CustomerTotals)
ORDER BY TotalPurchases DESC;

**Question 4** : Identify and handle missing values in the dataset using Copilot.

- Regions → "Unknown" if blank, inferred if customer had consistent regions elsewhere.

Products → "Unknown Product" if blank, inferred from sales/quantity patterns.

Sales → "NA" and blanks replaced with median values (Monitors ≈ 300, Phones ≈ 300, Laptops ≈ 300–400, Tablets ≈ 200–300).

Quantities → Converted text like "two" → 2. Filled blanks with median purchase quantity for that product.

**Question 5** : Task -

Identify duplicate rows

Remove duplicates using Copilot suggestion

- Duplicate Rows

Amit, West, Monitor, 300, 2, 2023-08-31 → appears more than once.

Riya, West, Phone, 100, 2, 2023-06-17 → repeated.

Sneha, East, Monitor, 300,, 2023-12-26 → occurs multiple times.

Karan, North, Laptop, 300, 4, 2023-06-12 → duplicated.

Rahul, East, Tablet, 200, 4, 2023-07-30 → repeated.

Priya, West, Tablet, 100, 2, 2023-03-27 → appears more than once.

Riya, East, Monitor, 200, 1, 2023-02-13 → duplicated.

Sneha, East, Phone, 400, 2, 2023-07-08 → repeated.

Rahul, West, Phone, 300, 3, 2023-06-07 → occurs multiple times.

Priya, South, Phone, 300, 2, 2023-01-07 → duplicated.

49 duplicate rows removed.

**Question 6** : Generate:

Column chart OR bar chart and Interpret which region has highest sales

- From the cleaned dataset, the South Region has the highest total sales, followed closely by the West Region.

South’s strength comes from consistent sales across Phones, Laptops, and Tablets.

West also performs strongly, especially in Laptops and Phones.

North and East trail behind, while Unknown (imputed values) sits in the middle.

**Question 7** : Create a pivot table showing total sales by region and product.

The South Region has the highest total sales, driven by strong performance across all product categories.

The West Region is the next strongest, especially in Laptops and Phones.

North and East trail behind, while Unknown sits in the middle due to imputed values

**Question 8** : Use Copilot to generate a new calculated column Total Sales.

- The Total Sales column now shows the revenue contribution per row.

You can use this new column in a pivot table to analyze sales by region and product.

For example, summing Total Sales by Region will confirm that the South Region has the highest overall sales.

**Question 9**: Summarize the dataset and find the key insights.

#Dataset Summary

Columns: Customer_Name, Region, Product, Sales, Quantity, Order_Date, Total Sales (calculated).

Rows: Transactions across multiple customers, regions, and products.

Products: Monitor, Laptop, Phone, Tablet, plus some “Unknown Product” entries (imputed).

Regions: North, South, East, West, Unknown (filled for missing values).

Sales Values: Standardized to numeric, with missing entries imputed using product-level medians.

Quantities: Converted to integers, blanks filled with product-level medians.

Duplicates

#Key Insights

South Region

Highest overall sales.

Strong performance across all product categories, especially Phones and Laptops.

West Region

Second highest sales.

Laptops and Phones drive most of the revenue.

North Region

Moderate sales, with Monitors and Tablets contributing steadily.

East Region

Lowest sales overall, fewer transactions compared to other regions.

Unknown Region

Represents imputed values; sits in the middle range of sales totals.

#Product-Level Insights
Laptops and Phones are the top revenue generators.

Monitors show consistent but smaller sales volumes.

Tablets contribute moderately, often in smaller quantities.

#Overall Takeaway

The South Region dominates sales, making it the most profitable geography.

West Region is a close competitor, especially in high-value products.

Laptops and Phones are the most important product categories for revenue.

Cleaning the dataset (handling missing values, imputations, and duplicates) ensures reliable insights for decision-making.

**Question 10** : Generate VBA code using Copilot for cleaning data (removing duplicates, filling null values,
format table).

In [ ]:
Sub CleanDataset()

    Dim ws As Worksheet
    Dim tbl As ListObject
    Dim rng As Range

    ' Use the active sheet
    Set ws = ActiveSheet

    ' Define the used range
    Set rng = ws.UsedRange

    ' Remove duplicates across all columns
    rng.RemoveDuplicates Columns:=Array(1, 2, 3, 4, 5, 6), Header:=xlYes

    ' Fill blanks in Region and Product with "Unknown"
    rng.Columns(2).SpecialCells(xlCellTypeBlanks).Value = "Unknown"
    rng.Columns(3).SpecialCells(xlCellTypeBlanks).Value = "Unknown Product"

    ' Fill blanks in Sales and Quantity with 0
    rng.Columns(4).SpecialCells(xlCellTypeBlanks).Value = 0
    rng.Columns(5).SpecialCells(xlCellTypeBlanks).Value = 0

    ' Format as a table
    On Error Resume Next
    Set tbl = ws.ListObjects("CleanedTable")
    On Error GoTo 0

    If tbl Is Nothing Then
        Set tbl = ws.ListObjects.Add(xlSrcRange, rng, , xlYes)
        tbl.Name = "CleanedTable"
    End If

    ' Apply a table style
    tbl.TableStyle = "TableStyleMedium9"

    MsgBox "Dataset cleaned: duplicates removed, blanks filled, and table formatted.", vbInformation

End Sub
